# **NetMHCpan wrapper development notebook**

### **Goals**:

Given HLA/Epitope table predict affinity values using NetMHCpan

Make smart match: A02 = A0201, A0202 etc

Given epitope table only predict affinity for best HLA

Don't mess affinity types: keep affinity rank and affinity value separate

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import os
import subprocess
import itertools
import resource
import xlrd
import regex as re

### **Config**

In [3]:
#resource.setrlimit(resource.RLIMIT_STACK, (resource.RLIM_INFINITY, resource.RLIM_INFINITY))


PATH_TO_NETMHCPAN = Path('/home/aalarkin/Work/netmhcpan_wrapper/netMHCpan-4.2bstatic.Linux/netMHCpan-4.2')
TMPDIR = Path('/home/aalarkin/Work/netmhcpan_wrapper/tmp')
PATH_TO_SUPERGROUPS = Path('/home/aalarkin/Work/netmhcpan_wrapper/datasets/mhc_supergroups.txt')
PATH_TO_MAPPING = Path('/home/aalarkin/Work/netmhcpan_wrapper/datasets/allele_nomenclature_mapping.txt')


ALPHABET = {'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y'}


### **Background datasets**:

1. datasets/mhc_supergroups.txt. Contains MHC supergroups to enable pan prediction for epitopes if only allele type is known i.e HLA-G01

Example: HLA-G01;HLA-G\*01:01,HLA-G\*01:02,HLA-G\*01:03,HLA-G\*01:04,HLA-G\*01:06,HLA-G\*01:07,HLA-G\*01:08,HLA-G*01:09

2. datasets/allele_nomenclature_mapping.txt. Contains standartized HLA labels, mapped to possible ways to refer to the same label. Enables netMHCpan prediction with inconsitent HLA format

Example: HLA-A\*02:429;HLA-A\*02:429,HLA-A02:429,HLA-A\*02429,HLA-A02429,A\*02:429,A02:429,A02429

### **Utilities**

1. Creating and validating peplists

In [4]:
def validate_peptide(seq):
    if not len(seq) > 0 or type(seq) != str:
        return(False)

    for a in list(seq):
        if a not in ALPHABET:
            return(False)
        
    return(True)


def generate_peplist(pep_array, output_file):

    with open(output_file, mode = 'w') as o:
        for s in pep_array:
            if validate_peptide(s) == True:
                o.write(f'{s}\n')
            else:
                raise ValueError(f'Peptide {s} is invalid')

    

In [5]:
peptides = ['WWWWWWTTTYYYYY', 'GLGLGIIIWWVV', 'WWWYYYKKKQRS']
generate_peplist(peptides, f'{TMPDIR}/pep_test.txt')
subprocess.run(f'cat {TMPDIR}/pep_test.txt', shell = True)

WWWWWWTTTYYYYY
GLGLGIIIWWVV
WWWYYYKKKQRS


CompletedProcess(args='cat /home/aalarkin/Work/netmhcpan_wrapper/tmp/pep_test.txt', returncode=0)

### **Core functions**

### Given HLA/Epitope table predict affinity values using NetMHCpan

In [6]:
def predict_affinnity(epitope, hla):
    if not validate_peptide(epitope):
        raise ValueError(f'Epitope validation for {epitope} failed')
    
    subprocess.run(f'echo "{epitope}" > {TMPDIR}/temp_epitope.txt', shell=True, check=True)
    res = str(subprocess.run(f'{PATH_TO_NETMHCPAN}/netMHCpan -a {hla} -p {TMPDIR}/temp_epitope.txt -xls -xlsfile {TMPDIR}/affinity_pred.xls -BA -mode 0' , shell=True, check=True, capture_output= True).stdout.decode('utf-8'))
   # print(res)
    subprocess.run(f'rm {TMPDIR}/temp_epitope.txt', shell=True, check=True)
    match = re.search(r'Score_BA\s+([\d.]+)\s+%Rank_BA\s+([\d.]+)\s+Aff\(nM\)\s+([\d.]+)(?:\s+<= [A-Z]+)?', res)

    if not match:
        lines = res.split('\n')
        for line in lines:
            if epitope in line:
                numbers = re.findall(r'\d+(?:\.\d+)?', line)
                if len(numbers) >= 3:
                    score_ba = float(numbers[-3])  
                    percent_rank_ba = float(numbers[-2]) 
                    affinity_nm = float(numbers[-1])  
                    return (score_ba, percent_rank_ba, affinity_nm)
        
        raise ValueError(f'Could not extract binding data for epitope {epitope}')


    score_ba = float(match.group(1))  
    percent_rank_ba = float(match.group(2)) 
    affinity_nm = float(match.group(3))

    return(score_ba, percent_rank_ba, affinity_nm)



In [7]:
predict_affinnity('WWWWWWWW', 'HLA-A01:01')


(0.024391, 53.667, 38402.22)

**1.2. Handling pd dataframes**

In [8]:
def predict_affinity_dataframe(df, epitope_colname, hla_colname):
    scores = []
    ranks = []
    affs = []

    for index, row in df.iterrows():

        try:
            res = predict_affinnity(row[epitope_colname], row[hla_colname])
            v1,v2,v3 = res
        except:
            v1,v2,v3 = np.nan, np.nan, np.nan
    
        scores.append(v1)
        ranks.append(v2)
        affs.append(v3)

  #  print(scores)
  #  print(ranks)
  #  print(affs)

    df['%Rank Score_BA'] = scores
    df['%Rank_BA'] = ranks
    df['Aff(nM)'] = affs

    return(df)

**1.3. Handling inconsistent labels**

In [9]:
def match_hla(df, hla_colname):
    mapping = {}
    matched_hlas = []

    with open(PATH_TO_MAPPING, mode='r') as mp:
        for i in mp:
            line = i.split(';')
            mapping[line[0]] = set(line[1].rstrip('\n').split(','))
   # print(mapping['HLA-A*68:01'])
    mapping_keys = set(mapping.keys())

    for query in df[hla_colname].tolist():
        if query in mapping_keys:  # matches standardized nomenclature i.e HLA-A*02:01
            matched_hlas.append(''.join(query.split('*')))
        else:
            found = False
            for h in mapping:
                if query in mapping[h]:
                    matched_hlas.append(''.join(h.split('*')))
                    found = True
                    break
            if not found:
                matched_hlas.append(np.nan)
    
  #  print(matched_hlas)
    df['matched_hla'] = matched_hlas
    return df



In [10]:
inconsistent_data = {
    "IVDSLTEMY": "HLA-A*01:01",
    "LIDGIFLRY": "HLA-A*0101",
    "GLLPSLLLLL": "HLA-A0201",
    "EVISVMKRR": "A6801",
    "FQKVITEY": "B*15:01",
    "SEEDFIRSL": "B*4104",
    "VMADRTRHL": "E0103",
    "ANADLEVKI": "HLA-C*0602"
}

df = pd.DataFrame(list(inconsistent_data.items()), columns=['epitope', 'hla'])
display(df)

,epitope,hla
0,IVDSLTEMY,HLA-A*01:01
1,LIDGIFLRY,HLA-A*0101
2,GLLPSLLLLL,HLA-A0201
3,EVISVMKRR,A6801
4,FQKVITEY,B*15:01
5,SEEDFIRSL,B*4104
6,VMADRTRHL,E0103
7,ANADLEVKI,HLA-C*0602


In [11]:
df = match_hla(df, 'hla')
display(df)

,epitope,hla,matched_hla
0,IVDSLTEMY,HLA-A*01:01,HLA-A01:01
1,LIDGIFLRY,HLA-A*0101,HLA-A01:01
2,GLLPSLLLLL,HLA-A0201,HLA-A02:01
3,EVISVMKRR,A6801,HLA-A68:01
4,FQKVITEY,B*15:01,HLA-B15:01
5,SEEDFIRSL,B*4104,HLA-B41:04
6,VMADRTRHL,E0103,HLA-E01:03
7,ANADLEVKI,HLA-C*0602,HLA-C06:02


In [12]:
df = predict_affinity_dataframe(df , 'epitope', 'matched_hla')
display(df)

,epitope,hla,matched_hla,%Rank Score_BA,%Rank_BA,Aff(nM)
0,IVDSLTEMY,HLA-A*01:01,HLA-A01:01,0.658625,0.013,40.19
1,LIDGIFLRY,HLA-A*0101,HLA-A01:01,0.675019,0.011,33.66
2,GLLPSLLLLL,HLA-A0201,HLA-A02:01,0.752222,0.131,14.60
3,EVISVMKRR,A6801,HLA-A68:01,0.834083,0.041,6.02
4,FQKVITEY,B*15:01,HLA-B15:01,0.563743,0.624,112.19
5,SEEDFIRSL,B*4104,HLA-B41:04,0.578451,0.147,95.69
6,VMADRTRHL,E0103,HLA-E01:03,0.384711,0.032,778.44
7,ANADLEVKI,HLA-C*0602,HLA-C06:02,0.128117,5.632,12501.21


### **Testing on actual data**

In [13]:
df = pd.read_csv('../datasets/chowell.txt', sep = '\t')[:100]
df = match_hla(df, 'hla')
df = predict_affinity_dataframe(df, 'antigen.epitope', 'matched_hla')
display(df)

,antigen.epitope,hla,immunogenicity,matched_hla,%Rank Score_BA,%Rank_BA,Aff(nM)
0,ITDFNIDTY,HLA-A*01:01,Positive,HLA-A01:01,0.732399,0.008,18.09
1,ATDALMTGF,HLA-A*01:01,Positive,HLA-A01:01,0.427019,0.133,492.51
2,SSNIMSESY,HLA-A*01:01,Positive,HLA-A01:01,0.464926,0.095,326.81
3,VSEKYTDMY,HLA-A*01:01,Positive,HLA-A01:01,0.710627,0.009,22.90
4,FTDWANKQY,HLA-A*01:01,Positive,HLA-A01:01,0.715700,0.009,21.67
...,...,...,...,...,...,...,...
95,IADMGHLKY,HLA-A*01:01,Negative,HLA-A01:01,0.676161,0.011,33.24
96,DSDLRNDSY,HLA-A*01:01,Negative,HLA-A01:01,0.556244,0.040,121.67
97,ITEDKTELY,HLA-A*01:01,Negative,HLA-A01:01,0.718329,0.009,21.07
98,ALQEALTEHY,HLA-A*01:01,Negative,HLA-A01:01,0.259031,0.589,3032.42


### **2. Given epitope table, predict affinity for best hla**

In [36]:
def pan_hla_matching_prediction(df, epitope_column, helper_column = None):


    allele_column = []
    score_ba_column = []
    percent_rank_ba_column = []
    affinity_nm_column = []

    supergroups = {}

    with open(PATH_TO_SUPERGROUPS, mode = 'r') as f:

        for string in f:

            sg_label, representative, other_alleles = string.split(';')

            supergroups[sg_label] = {'representative': representative}

            if other_alleles == '0':
                supergroups[sg_label]['alleles'] = None
            else:
                supergroups[sg_label]['alleles'] = other_alleles.split(',')


    for index,row in df.iterrows():

        best_rank = 1000
        best_aff = float('inf')
        best_allele = None
        best_score_ba = None


        #Case 1 - no helper column provided or helper_column[index] is empty  or invalid--> 2 rounds of prediction

        if (not helper_column) or (not row[helper_column] in supergroups):
            best_supergroup = None
            best_supergroup_aff = float('inf')
            best_supergroup_rank = 1000
            best_supergroup_score_ba = None

            #1st round - finding best supergroup
            for sg in supergroups:
                repr = supergroups[sg]['representative']


                if '*' in repr:
                    repr = ''.join(repr.split('*'))

                    try:
                        score_ba, percent_rank_ba, affinity_nm = predict_affinnity(row[epitope_column], repr )

                        if percent_rank_ba < best_supergroup_rank and affinity_nm < best_supergroup_aff:

                            best_supergroup_rank = percent_rank_ba
                            best_supergroup_aff = affinity_nm
                            best_supergroup = sg
                            best_supergroup_score_ba = score_ba

                    except:
                        continue


            #2nd round - finding best allele in supergroup



            #best supergroup is singlet

            if not supergroups[best_supergroup]['alleles']:

                best_rank = best_supergroup_rank
                best_aff = best_supergroup_aff
                best_allele = a
                best_score_ba = best_supergroup_score_ba


            #best supergroup is not singlet

            else:

                alleles = supergroups[best_supergroup]['alleles']

                for a in alleles:

                    if '*' in a:      
                        a = ''.join(a.split('*'))

                    try:
                        score_ba, percent_rank_ba, affinity_nm = predict_affinnity(row[epitope_column], a )


                        if percent_rank_ba < best_rank and affinity_nm < best_aff:

                            best_rank = percent_rank_ba
                            best_aff = affinity_nm
                            best_allele = a
                            best_score_ba = score_ba
                    except:
                        continue

                











        #Case 2 - helper column[index] is valid --> 1 round of prediction

        else:
            repr = supergroups[row[helper_column]]['representative']
            others = supergroups[row[helper_column]]['alleles']

            if others:
                alleles = others
            else:
                alleles = [repr]

         #   print(alleles)
            for a in alleles:

                if '*' in a:       #edge case for human mhc-standartized label has *, netmchpan label does not
                    a = ''.join(a.split('*'))
                    #print(a)

                    try:
                        score_ba, percent_rank_ba, affinity_nm = predict_affinnity(row[epitope_column], a )




                        if percent_rank_ba < best_rank and affinity_nm < best_aff:

                            best_rank = percent_rank_ba
                            best_aff = affinity_nm
                            best_allele = a
                            best_score_ba = score_ba

                    except:
                        continue



        allele_column.append(best_allele)
        score_ba_column.append(best_score_ba)
        percent_rank_ba_column.append(best_rank)
        affinity_nm_column.append(best_aff)





    df['Allele'] = allele_column           
    df['%Rank Score_BA'] = score_ba_column
    df['%Rank_BA'] = percent_rank_ba_column
    df['Aff(nM)'] = affinity_nm_column

    return(df)

                                  
                
                



        

**Testing on dataframe with all families known**

In [25]:
data = {
 #   "IVDSLTEMY": "HLA-A01", #true answer - hla a0101
    "LIDGIFLRY": "HLA-A01", #true answer - hla a0101
    "FQKVITEY": "HLA-B15",    #true answer - hla b1501
    "VMADRTRHL": "HLA-E01",      #true answer - e0103
    "ANADLEVKI": "HLA-C06"       #true answer - HLA-C*0602
}

In [26]:
df = pd.DataFrame(list(data.items()), columns=['epitope', 'family'])
display(df)


,epitope,family
0,LIDGIFLRY,HLA-A01
1,FQKVITEY,HLA-B15
2,VMADRTRHL,HLA-E01
3,ANADLEVKI,HLA-C06


In [37]:
df = pan_hla_matching_prediction(df, 'epitope', 'family')

In [38]:
display(df)

,epitope,family,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
0,LIDGIFLRY,HLA-A01,HLA-A01:191,0.715868,0.008,21.63
1,FQKVITEY,HLA-B15,HLA-B15:74,0.702537,0.433,24.99
2,VMADRTRHL,HLA-E01,HLA-E01:01,0.384711,0.032,778.44
3,ANADLEVKI,HLA-C06,HLA-C06:03,0.211832,4.290,5053.32


checked these prediction via cmd netmhcpan - these alleles actually more affine that those one from dataframe

**Testing on dataframe with some families known**

In [40]:
data = {
  #  "LIDGIFLRY": np.nan, 
  #  "FQKVITEY": "HLA-B15",    #true answer - hla b1501
    "LIDGIFLRY": None,          #true answer - hla a0101  
    "VMADRTRHL": "HLA-E01",      #true answer - e0103
    "ANADLEVKI": "HLA-C06",       #true answer - HLA-C*0602
                    
}



In [42]:
df = pd.DataFrame(list(data.items()), columns=['epitope', 'family'])
display(df)


,epitope,family
0,LIDGIFLRY,NaN
1,VMADRTRHL,HLA-E01
2,ANADLEVKI,HLA-C06


In [ ]:
df = pan_hla_matching_prediction(df, 'epitope', 'family')

In [44]:
display(df)

,epitope,family,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
0,LIDGIFLRY,NaN,HLA-A80:03,0.725787,0.009,19.43
1,VMADRTRHL,HLA-E01,HLA-E01:01,0.384711,0.032,778.44
2,ANADLEVKI,HLA-C06,HLA-C06:03,0.211832,4.290,5053.32


**Testing on pure epitope dataframe**



In [48]:
epitopes = ['LIDGIFLRY', 'VMADRTRHL', 'ANADLEVKI' ]
df = pd.DataFrame(epitopes, columns=['epitope'])
display(df)

df = pan_hla_matching_prediction(df, 'epitope')


,epitope
0,LIDGIFLRY
1,VMADRTRHL
2,ANADLEVKI


In [49]:
display(df)

,epitope,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
0,LIDGIFLRY,HLA-A80:03,0.725787,0.009,19.43
1,VMADRTRHL,HLA-C01:102,0.742341,0.090,16.25
2,ANADLEVKI,HLA-C15:57,0.261550,1.929,2950.89


also checked manually - affinities lower then dataset values and then family-restricted best allele